0) Imports + Paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# ---- directories ----
BASE_DIR = Path(r"D:\Thesis\Old\Codes\New\feature_engineering_outputs")

FEATURES_PATH = BASE_DIR / "features_wide_10y_100tickers.parquet"
TARGETS_PATH  = BASE_DIR / "targets_wide_10y_100tickers.parquet"

PANEL_DIR = Path(r"D:\Thesis\Old\Codes\New\panel_data")
PANEL_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path(r"D:\Thesis\Old\Codes\New\results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = RESULTS_DIR / "model_results.csv"


1) Feature sets (locked)

In [2]:
RISK_FEATURES = [
    "semi_variance_63d",
    "volatility_21d",
    "downside_volatility_63d",
    "kurtosis_63d",
    "volatility_126d",
]

RETURN_FEATURES = [
    "ret_1d", "ret_5d", "ret_21d",
    "lag1", "lag3", "lag6",
    "ma_ratio_21_63", "ma_ratio_63_126",
    "zscore_price_63d",
    "alpha_FF_6m", "alpha_FF_12m",
    "cross_sectional_rank_momentum",
    "relative_return_rank",
    "volatility_63d",
]


2) Load wide data (Features + Targets)

In [3]:
print("Loading wide parquet files...")
Features = pd.read_parquet(FEATURES_PATH).sort_index()
Targets  = pd.read_parquet(TARGETS_PATH).sort_index()

print("Features:", Features.shape)
print("Targets :", Targets.shape)

# sanity
print("Target base names:", sorted({c.split("__",1)[0] for c in Targets.columns if "__" in c}))


Loading wide parquet files...
Features: (2515, 7140)
Targets : (2515, 600)
Target base names: ['target_class_topN', 'target_class_updown', 'target_cvar_next1m', 'target_excess_return_next1m', 'target_return_next1m', 'target_volatility_next1m']


3) Recreate correct lag1/lag3/lag6 in FEATURES (fix)

This creates a new features parquet with corrected lag columns.

In [4]:
OUT_FEATURES_FIXED = BASE_DIR / "features_wide_10y_100tickers_FIXEDLAGS.parquet"

ret_cols = [c for c in Features.columns if c.startswith("ret_1d__")]
if not ret_cols:
    raise ValueError("No 'ret_1d__TICKER' columns found. Can't build lags.")

tickers = sorted({c.split("__", 1)[1] for c in ret_cols})
print("Tickers detected:", len(tickers))

new_cols = {}
for t in tickers:
    s = Features[f"ret_1d__{t}"]
    new_cols[f"lag1__{t}"] = s.shift(1)
    new_cols[f"lag3__{t}"] = s.shift(3)
    new_cols[f"lag6__{t}"] = s.shift(6)

lags_df = pd.DataFrame(new_cols, index=Features.index)

# overwrite old broken lag columns if they exist
for col in lags_df.columns:
    if col in Features.columns:
        Features.drop(columns=[col], inplace=True)

Features_fixed = pd.concat([Features, lags_df], axis=1)

# quick sanity (pick first ticker)
t0 = tickers[0]
print("Lag NaN rates sample:")
print(Features_fixed[[f"lag1__{t0}", f"lag3__{t0}", f"lag6__{t0}"]].isna().mean())

Features_fixed.to_parquet(OUT_FEATURES_FIXED)
print("✅ Saved:", OUT_FEATURES_FIXED)


Tickers detected: 100
Lag NaN rates sample:
lag1__AAPL    0.000795
lag3__AAPL    0.001590
lag6__AAPL    0.002783
dtype: float64
✅ Saved: D:\Thesis\Old\Codes\New\feature_engineering_outputs\features_wide_10y_100tickers_FIXEDLAGS.parquet


4) Panel builder (wide → panel)

In [5]:
def build_panel_from_wide(features_wide: pd.DataFrame,
                          targets_wide: pd.DataFrame,
                          feature_list: list[str],
                          target_prefix: str) -> pd.DataFrame:
    # find target columns
    tcols = [c for c in targets_wide.columns if c.startswith(target_prefix)]
    if not tcols:
        raise ValueError(f"No target columns found with prefix: {target_prefix}")

    # tickers from targets
    tickers = sorted({c.split("__", 1)[1] for c in tcols})

    # Y long
    Y = targets_wide[tcols].copy()
    Y.columns = [c.split("__", 1)[1] for c in Y.columns]  # ticker only
    Y_long = Y.stack().rename("y").reset_index()
    Y_long.columns = ["Date", "Ticker", "y"]

    # X long (merge feature-by-feature)
    frames = []
    for feat in feature_list:
        cols = [c for c in features_wide.columns if c.startswith(feat + "__")]
        if not cols:
            print(f"WARNING: feature not found in wide data: {feat}")
            continue
        Xf = features_wide[cols].copy()
        Xf.columns = [c.split("__", 1)[1] for c in Xf.columns]
        Xf = Xf.reindex(columns=tickers)
        tmp = Xf.stack().rename(feat).reset_index()
        tmp.columns = ["Date", "Ticker", feat]
        frames.append(tmp)

    if not frames:
        raise ValueError("No features matched feature_list.")

    X_long = frames[0]
    for f in frames[1:]:
        X_long = X_long.merge(f, on=["Date", "Ticker"], how="left")

    panel = Y_long.merge(X_long, on=["Date", "Ticker"], how="inner")
    panel = panel.set_index(["Date", "Ticker"]).sort_index()
    return panel


5) Build & save the two panels

In [6]:
# load the fixed features
Features_fixed = pd.read_parquet(OUT_FEATURES_FIXED).sort_index()

# build return panel (target = next-month return)
panel_return = build_panel_from_wide(
    Features_fixed, Targets, RETURN_FEATURES,
    target_prefix="target_return_next1m__"
)

# build risk panel (target = next-month CVaR)
panel_risk = build_panel_from_wide(
    Features_fixed, Targets, RISK_FEATURES,
    target_prefix="target_cvar_next1m__"
)

print("panel_return:", panel_return.shape)
print("panel_risk  :", panel_risk.shape)

# save
(panel_return).to_parquet(PANEL_DIR / "panel_return.parquet")
(panel_risk).to_parquet(PANEL_DIR / "panel_risk.parquet")
print("✅ Saved panels to:", PANEL_DIR)


panel_return: (249300, 15)
panel_risk  : (243100, 6)
✅ Saved panels to: D:\Thesis\Old\Codes\New\panel_data


6) Cleaning functions (correct + minimal)
Return panel cleaning (fix NaNs without killing the dataset)

In [7]:
def clean_return_panel(panel: pd.DataFrame) -> pd.DataFrame:
    panel = panel.copy()

    # ensure target exists
    panel = panel.dropna(subset=["y"])

    feature_cols = [c for c in panel.columns if c != "y"]

    # forward fill features within each ticker (no look-ahead)
    panel[feature_cols] = (
        panel[feature_cols]
        .groupby(level="Ticker", group_keys=False)
        .apply(lambda df: df.ffill())
    )

    # drop remaining NaNs (early rolling window only)
    panel = panel.dropna()
    return panel


Risk panel cleaning (simple dropna is fine)

In [8]:
def clean_risk_panel(panel: pd.DataFrame) -> pd.DataFrame:
    return panel.dropna().copy()


In [9]:
panel_return_clean = clean_return_panel(panel_return)
panel_risk_clean   = clean_risk_panel(panel_risk)

print("panel_return_clean:", panel_return_clean.shape)
print("panel_risk_clean  :", panel_risk_clean.shape)


panel_return_clean: (220653, 15)
panel_risk_clean  : (236800, 6)


7) Time split + model helpers

In [10]:
def prepare_xy(panel: pd.DataFrame):
    X = panel.drop(columns=["y"])
    y = panel["y"]
    return X, y

def time_split(panel: pd.DataFrame, train_ratio=0.7):
    dates = panel.index.get_level_values("Date").unique().sort_values()
    split_date = dates[int(len(dates) * train_ratio)]
    train_mask = panel.index.get_level_values("Date") <= split_date
    test_mask  = panel.index.get_level_values("Date") > split_date
    return panel.loc[train_mask], panel.loc[test_mask], split_date

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


8) Results table + logger (reusable)

In [11]:
# initialize results file once
if not RESULTS_PATH.exists():
    pd.DataFrame(columns=[
        "task", "model", "target", "split_date",
        "rmse", "n_train", "n_test", "notes"
    ]).to_csv(RESULTS_PATH, index=False)
    print("✅ Initialized:", RESULTS_PATH)

def log_result(task, model, target, split_date, rmse_val, n_train, n_test, notes=""):
    row = pd.DataFrame([{
        "task": task,
        "model": model,
        "target": target,
        "split_date": split_date,
        "rmse": rmse_val,
        "n_train": n_train,
        "n_test": n_test,
        "notes": notes
    }])
    row.to_csv(RESULTS_PATH, mode="a", header=False, index=False)


9) Baseline 1 — Ridge (Return + Risk)
Return (Ridge needs scaling)

In [12]:
train_ret, test_ret, split_date_ret = time_split(panel_return_clean)

X_train, y_train = prepare_xy(train_ret)
X_test,  y_test  = prepare_xy(test_ret)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)
pred = ridge.predict(X_test_sc)

rmse_ret = rmse(y_test, pred)
print("Ridge RETURN RMSE:", rmse_ret)

log_result("return", "Ridge", "target_return_next1m", split_date_ret, rmse_ret, len(train_ret), len(test_ret), "baseline linear")


Ridge RETURN RMSE: 0.08840710308258529


Risk (also scale)

In [13]:
train_risk, test_risk, split_date_risk = time_split(panel_risk_clean)

X_train, y_train = prepare_xy(train_risk)
X_test,  y_test  = prepare_xy(test_risk)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)
pred = ridge.predict(X_test_sc)

rmse_risk = rmse(y_test, pred)
print("Ridge RISK(CVaR) RMSE:", rmse_risk)

log_result("risk", "Ridge", "target_cvar_next1m", split_date_risk, rmse_risk, len(train_risk), len(test_risk), "baseline linear")


Ridge RISK(CVaR) RMSE: 0.011871749741993183


10) Baseline 2 — Random Forest (Return + Risk)

RF does not need scaling.

Return

In [14]:
train_ret, test_ret, split_date_ret = time_split(panel_return_clean)

X_train, y_train = prepare_xy(train_ret)
X_test,  y_test  = prepare_xy(test_ret)

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

rmse_rf_ret = rmse(y_test, pred)
print("RF RETURN RMSE:", rmse_rf_ret)

log_result("return", "RandomForest", "target_return_next1m", split_date_ret, rmse_rf_ret, len(train_ret), len(test_ret),
           "n_estimators=300, max_depth=10, min_samples_leaf=50")


RF RETURN RMSE: 0.089308642833404


Risk

In [15]:
train_risk, test_risk, split_date_risk = time_split(panel_risk_clean)

X_train, y_train = prepare_xy(train_risk)
X_test,  y_test  = prepare_xy(test_risk)

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

rmse_rf_risk = rmse(y_test, pred)
print("RF RISK(CVaR) RMSE:", rmse_rf_risk)

log_result("risk", "RandomForest", "target_cvar_next1m", split_date_risk, rmse_rf_risk, len(train_risk), len(test_risk),
           "n_estimators=300, max_depth=10, min_samples_leaf=50")


RF RISK(CVaR) RMSE: 0.012434965385100714


11) View leaderboard

In [16]:
results = pd.read_csv(RESULTS_PATH)
results.sort_values(["task", "rmse"])


,task,model,target,split_date,rmse,n_train,n_test,notes
0,return,Ridge,target_return_next1m,2022-09-22,0.088407,153831,66822,baseline linear model
2,return,Ridge,target_return_next1m,2022-09-22,0.088407,153831,66822,baseline linear
4,return,RandomForest,target_return_next1m,2022-09-22,0.089309,153831,66822,"n_estimators=300, max_depth=10, min_samples_le..."
1,risk,Ridge,target_cvar_next1m,2022-07-29,0.011872,165800,71000,baseline linear model
3,risk,Ridge,target_cvar_next1m,2022-07-29,0.011872,165800,71000,baseline linear
5,risk,RandomForest,target_cvar_next1m,2022-07-29,0.012435,165800,71000,"n_estimators=300, max_depth=10, min_samples_le..."


What this means scientifically

Your pipeline is deterministic and stable

Same split + same model → same RMSE

That’s exactly what examiners want to see

You can confidently say:

“Results are fully reproducible under identical experimental settings.”

What you should do now (very simple)
1️⃣ Keep only ONE Ridge row per task

You don’t need to delete anything yet, but for presentation you’ll deduplicate later.

If you want a clean working view now:

In [17]:
results = pd.read_csv(RESULTS_PATH)

results_clean = (
    results
    .sort_values("rmse")
    .drop_duplicates(subset=["task", "model", "target"], keep="first")
)

results_clean


,task,model,target,split_date,rmse,n_train,n_test,notes
1,risk,Ridge,target_cvar_next1m,2022-07-29,0.011872,165800,71000,baseline linear model
5,risk,RandomForest,target_cvar_next1m,2022-07-29,0.012435,165800,71000,"n_estimators=300, max_depth=10, min_samples_le..."
0,return,Ridge,target_return_next1m,2022-09-22,0.088407,153831,66822,baseline linear model
4,return,RandomForest,target_return_next1m,2022-09-22,0.089309,153831,66822,"n_estimators=300, max_depth=10, min_samples_le..."
